# Deep Learning
<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/marcinsawinski/UEP_KIE_DL_CODE2024/blob/main/dl03_tuning.ipynb" target="_parent">
      <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
    </a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/marcinsawinski/UEP_KIE_DL_CODE2024/blob/main/dl03_tuning.ipynb">
      <img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/>
    </a>
  </td>
  <td>
    <a target="_blank" href="https://studiolab.sagemaker.aws/import/github/marcinsawinski/UEP_KIE_DL_CODE2024/blob/main/dl03_tuning.ipynb">
      <img src="https://studiolab.sagemaker.aws/studiolab.svg" alt="Open in SageMaker Studio Lab"/>
    </a>
  </td>
</table>

# Tasks
1. Install and connect to [WandB](https://wandb.ai) and Run a training simulation
2. Build NN model for classification on Fashion MNIST dataset. Log training to WandB in own project.
3. Create hyperparameter search with WandB sweep.
4. Experiment with Optimizers (SGD, ADAM) and optimizer parameters - number of epochs, learnign rate.
5. Experiment with network depth and number of neurons in layers
6. Experiment with Schedulers
7. Experiment with Dropout layers
8. Experiment with Batch Normalization layers
9. Try early stopping.
10. Find optimal setup. Retrain with the optimal setup and Log training to WandB in project called DL25-FMNIST


## Task1 - Install and connect to [WandB](https://wandb.ai)
1. Run `pip install wandb -qU`in terminal or `!pip install wandb -qU` in jupyter cell
2. Login using `wandb login` (terminal) or !wandb login' (jupyter cell). Alternatively you can run
```
    import wandb
    wandb.login()
```
3. Update wandb.init code with your own project name e.g. student_surname_firstname. The entity can be "uep-kie-dl25" if you are already assigned.
4. Run simulation and review results on the [WandB](https://wandb.ai) website.


You will find WandB detailed example here:
https://colab.research.google.com/github/wandb/examples/blob/master/colabs/intro/Intro_to_Weights_%26_Biases.ipynb


In [1]:
# your code for installation and login goes here

In [3]:
import random
import time
import wandb
user = "kowalski_jan"
cfg = {
    "learning_rate": 0.02,
    "architecture": "ANN",
    "dataset": "dummy",
    "epochs": 5,
}
name = f"{cfg['architecture']}_{cfg['dataset']}_lr{cfg['learning_rate']}_ep{cfg['epochs']}_{time.strftime('%m%d-%H%M')}"
project = (f"student_{user}_dummy")
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="uep-kie-dl25",
    # Set the wandb project where this run will be logged.
    project=project,
    # Track hyperparameters and run metadata.
    name=name,
    config=cfg,
)


# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()

CommError: entity uep-kie-dl25 not found during upsertBucket

# Task 2 - Build NN model for classification on Fashion MNIST dataset. Log training to WandB in own project.

Create a basic NN model:
- Flatten data with `nn.Flatten()` layer
- 3 linear layers with 784, 128 and 64 inputs. outputs are 128,64 and 10 e.g. `nn.Linear(64, 10)`
- ReLU activation `nn.ReLU()`  (no activation for output layer)
- use cross entropy loss as criterion `nn.CrossEntropyLoss()`
- use  SGD optimizer `optim.SGD(model.parameters(), lr=config.lr)`


https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W%26B.ipynb

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math
import time
import wandb

In [9]:
sweep_config = {
    'method': 'random', # or 'grid', 'bayes'
    'metric': {'name': 'test/accuracy', 'goal': 'maximize'},
    'parameters': {
        'epochs': {'values': [5, 10]},
        'batch_size': {'values': [32, 64, 128]},
        'learning_rate': {'values': [1e-2, 1e-3, 1e-4]},
        'optimizer': {'values': ['adam', 'sgd']},
        'n_layers': {'values': [1, 2, 3]},
        'n_neurons': {'values': [128, 256, 512]},
        'dropout': {'values': [0.2, 0.4, 0.5]},
        'use_batchnorm': {'values': [True, False]},
        'scheduler': {'values': [True, False]}
    }
}

In [11]:
user = "franciszek_biskup" # your name here
cfg = {
    "learning_rate": 0.001,
    "n_layers": 2,
    "n_neurons": 512,
    "dropout": 0.2,
    "use_batchnorm": True,
    "optimizer": "adam",
    "epochs": 10,
    "batch_size": 64,
    "scheduler": False
}
#name = f"{cfg['dataset']}_lr{cfg['learning_rate']}_ep{cfg['epochs']}_{time.strftime('%m%d-%H%M')}"
#f"{cfg['architecture']}_{cfg['dataset']}_lr{cfg['learning_rate']}_ep{cfg['epochs']}_{time.strftime('%m%d-%H%M')}"
project = f"student_{user}_FMNIST"

In [14]:
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="UEP-DL26",
    # Set the wandb project where this run will be logged.
    project="DL26-FMNIST",
    # Track hyperparameters and run metadata.
    #name=name,
    config=cfg,
)

config = wandb.config

# 1. Device configuration
device = torch.device("mps" if torch.cuda.is_available() else "cpu")

# 2. Transformations
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

# 3. Load the dataset
train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size * 4)
n_steps_per_epoch = math.ceil(len(train_loader.dataset) / wandb.config.batch_size)


class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.5):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_acc = None
        self.early_stop = False

    def __call__(self, val_acc):
        if self.best_acc is None:
            self.best_acc = val_acc
        elif val_acc < self.best_acc + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_acc = val_acc
            self.counter = 0

# 4. Define the model
class FashionClassifier(nn.Module):
    def __init__(self, n_layers, n_neurons, dropout_rate, use_bn):
        super(FashionClassifier, self).__init__()
        layers = [nn.Flatten()]
        in_features = 28 * 28

        for _ in range(n_layers):
            layers.append(nn.Linear(in_features, n_neurons))
            if use_bn:
                layers.append(nn.BatchNorm1d(n_neurons))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_features = n_neurons

        layers.append(nn.Linear(in_features, 10))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def train():
    # Initialize WandB for the sweep run
    run = wandb.init()
    config = wandb.config

    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

    # Data setup
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
    test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size * 4)

    # Model, Loss, Optimizer
    model = FashionClassifier(config.n_layers, config.n_neurons, config.dropout, config.use_batchnorm).to(device)
    criterion = nn.CrossEntropyLoss()

    if config.optimizer == "adam":
        optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
    else:
        optimizer = optim.SGD(model.parameters(), lr=config.learning_rate, momentum=0.9)

    # Optional Scheduler
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1) if config.scheduler else None

    early_stopper = EarlyStopping(patience=2)

    # Training Loop
    for epoch in range(config.epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if scheduler: scheduler.step()

        # Evaluation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        avg_loss = total_loss / len(train_loader)

        wandb.log({"train/loss": avg_loss, "test/accuracy": accuracy, "epoch": epoch})
        print(f"Epoch {epoch}: Acc {accuracy:.2f}%")

        # Early Stopping check
        early_stopper(accuracy)
        if early_stopper.early_stop:
            print("Early stopping triggered.")
            break

# 3. Execution
# sweep_id = wandb.sweep(sweep_config, project="student_franciszek_biskup_FMNIST")
# wandb.agent(sweep_id, function=train, count=10)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 214kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.98MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 2.94MB/s]


In [15]:
# 3. Execution
sweep_id = wandb.sweep(sweep_config, project="student_franciszek_biskup_FMNIST")
wandb.agent(sweep_id, function=train, count=10)

Create sweep with ID: tz6zw9ea
Sweep URL: https://wandb.ai/UEP-DL26/student_franciszek_biskup_FMNIST/sweeps/tz6zw9ea


wandb: Agent Starting Run: x5xc0a1s with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.4
wandb: 	epochs: 5
wandb: 	learning_rate: 0.001
wandb: 	n_layers: 3
wandb: 	n_neurons: 256
wandb: 	optimizer: sgd
wandb: 	scheduler: False
wandb: 	use_batchnorm: False
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Exception in thread Exception in thread IntMsgThr:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/wandb/sdk/wandb_run.py", line 347, in check_internal_messages
    self._loop_check_status(
  File "/usr/local/lib/python3.12/dist-packages/wandb/sdk/wandb_run.py", line 236, in _loop_check_status
    local_handle = request()
                   ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/wandb/sdk/interface/interface.py", line 1000, in deliver_internal_messages
    return self._deliver_internal_messages(internal_message)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/wandb/sdk/interface/interface_shared.py", line 488, in _deliver_internal_messages
ChkStopThr    return self._deliver(record

Epoch 0: Acc 72.03%
Epoch 1: Acc 75.55%
Epoch 2: Acc 79.27%
Epoch 3: Acc 81.09%
Epoch 4: Acc 82.13%


epoch,▁▃▅▆█
test/accuracy,▁▃▆▇█
train/loss,█▃▂▁▁
epoch,4
test/accuracy,82.13
train/loss,0.55011


...

Epoch 0: Acc 72.52%
Epoch 1: Acc 75.67%
Epoch 2: Acc 75.80%
Epoch 3: Acc 75.93%
Early stopping triggered.


epoch,▁▃▆█
test/accuracy,▁▇██
train/loss,█▂▁▁
epoch,3
test/accuracy,75.93
train/loss,0.85635


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: c4y1z38s with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.5
wandb: 	epochs: 10
wandb: 	learning_rate: 0.01
wandb: 	n_layers: 2
wandb: 	n_neurons: 128
wandb: 	optimizer: adam
wandb: 	scheduler: True
wandb: 	use_batchnorm: True
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 0: Acc 83.79%
Epoch 1: Acc 84.11%
Epoch 2: Acc 86.44%
Epoch 3: Acc 86.63%
Epoch 4: Acc 86.70%
Early stopping triggered.


epoch,▁▃▅▆█
test/accuracy,▁▂▇██
train/loss,█▄▂▁▁
epoch,4
test/accuracy,86.7
train/loss,0.40014


In [16]:
with wandb.init(project="DL26-FMNIST", entity="uep-kie-dl25", config=optimal_cfg) as run:
    # Run the same training logic as above one last time
    train()

NameError: name 'optimal_cfg' is not defined

# Task 3 - Create hyperparameter search with WandB sweep.
Follow instructions available from WandB

https://colab.research.google.com/github/wandb/examples/blob/master/colabs/pytorch/Organizing_Hyperparameter_Sweeps_in_PyTorch_with_W%26B.ipynb

In [ ]:
# Experiment with sweeps

# Remaining tasks
4. Experiment with Optimizers (SGD, ADAM) and optimizer parameters - number of epochs, learning rate.
5. Experiment with network depth and number of neurons in layers
6. Experiment with Schedulers
7. Experiment with Dropout layers
8. Experiment with Batch Normalization layers
9. Try early stopping.
10. Find optimal setup. Retrain with the optimal setup and Log training to WandB in project called DL25-FMNIST. use your name as job name

**Optimizers**

Available [Optimizers](https://pytorch.org/docs/stable/optim.html):
```import torch.optim as optim
optimizer = optim.SGD(model.parameters(), lr=config.lr)
optimizer = optim.Adam(model.parameters(), lr=config.lr)
```
**Dropout**

Dropout layers: `nn.Dropout(0.5)` - add after the ReLU activations.

**Batch Normalization**

Batch Normalization layers 'nn.BatchNorm1d(64)'  - add right after Linear layers and before the activation functions.


**Scheduler**

```
optimizer = optim.SGD(model.parameters(), lr=0.01)
scheduler = ExponentialLR(optimizer, gamma=0.9)

for epoch in range(20):
    for input, target in dataset:
        optimizer.zero_grad()
        output = model(input)
        loss = loss_fn(output, target)
        loss.backward()
        optimizer.step()
    scheduler.step()
```



In [ ]:
# your code here